# Day 47: Image Data EDA Techniques  
## Step-by-Step Exploration with PIL for Computer Vision  
### Day 47 of 369-day Python & AI Learning Path 

## Introduction  

Welcome to Day 47 of your Python & AI Learning Path! Today we dive deeper into the fascinating world of **image data exploration** using the Python Imaging Library (PIL), now maintained as Pillow. While we previously explored structured datasets like MNIST and CIFAR-10, this notebook focuses on practical, real-world techniques for analyzing any image dataset you might encounter in production computer vision projects.

Images are complex, high-dimensional data sources that require specialized exploratory techniques. Unlike tabular data where each column has clear meaning, images contain millions of pixel values arranged in spatial patterns that convey visual information. Understanding the distribution of these pixels, the color channels, image dimensions, and quality issues is crucial before feeding data into deep learning models.

In this comprehensive guide, we'll use PIL Python's standard imaging library to load, inspect, and analyze image properties at scale. You'll learn how to detect corrupted files, identify outliers in resolution or aspect ratio, analyze color distributions across RGB channels, and assess overall image quality. These skills are essential whether you're building medical imaging classifiers, autonomous vehicle perception systems, or content moderation tools.

By the end of this notebook, you'll have a robust toolkit for image EDA that you can apply to any computer vision project. Let's unlock the pixels!

## Table of Contents  

1. [Introduction to Image Data EDA and the Role of PIL](#1-introduction-to-image-data-eda-and-the-role-of-pil)
2. [Loading Images Using PIL](#2-loading-images-using-pil)
3. [Basic Image Properties](#3-basic-image-properties)
4. [Visualizing Individual Images and Creating Grids](#4-visualizing-individual-images-and-creating-grids)
5. [Image Statistics Analysis](#5-image-statistics-analysis)
6. [Color Analysis](#6-color-analysis)
7. [Image Quality and Issues Detection](#7-image-quality-and-issues-detection)
8. [Aspect Ratio and Resolution Analysis](#8-aspect-ratio-and-resolution-analysis)
9. [Batch Image Analysis](#9-batch-image-analysis)
10. [Key Insights and Recommendations](#10-key-insights-and-recommendations)
11. [?Hands-On Exercises](#-hands-on-exercises)
12. [Solutions & Key Insights](#solutions--key-insights)

## 1. Introduction to Image Data EDA and the Role of PIL  

### Why Image EDA Matters

Before training computer vision models, you must understand your image data:

- **Dimension Consistency** Models require fixed-size inputs; inconsistent resolutions need resizing strategies
- **Color Space Variations** RGB vs. Grayscale vs. CMYK affect preprocessing pipelines
- **Quality Issues** Blurry, corrupted, or mislabeled images hurt model performance
- **Class Imbalance** Some categories may have fewer or lower-quality images
- **Aspect Ratio Distribution** Extreme ratios may require special handling (cropping, padding)

### Why PIL/Pillow?

PIL (Python Imaging Library) is the standard for Python image processing:
- **Lightweight & Fast** Efficient for batch operations
- **Format Support** Handles JPEG, PNG, BMP, TIFF, and 30+ formats
- **Rich Metadata** Access EXIF data, color profiles, and properties
- **Integration** Works seamlessly with NumPy, Matplotlib, and OpenCV
- **Preprocessing** Resize, rotate, convert color spaces, apply filters

Let's begin our exploration! 

In [ ]:
# Essential imports for image data analysis
from PIL import Image, ImageStat, ImageFilter, ImageOps
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("? All libraries imported successfully!")
print("Ready to explore image data with PIL!")

## 2. Loading Images Using PIL  

Let's start by loading images and understanding the basic PIL Image object. We'll create a sample dataset using images from a public source or generate synthetic data for demonstration.

In [ ]:
# Download sample images for demonstration
import urllib.request
import io

# Create a directory for sample images
os.makedirs('sample_images', exist_ok=True)

# URLs for sample images (cats and dogs from public datasets)
image_urls = [
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg", "cat_01.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg", "cat_02.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/1/15/White_Persian_Cat.jpg/1200px-White_Persian_Cat.jpg", "cat_03.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg", "dog_01.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/8/8a/Golden_Retriever_9-year_old.jpg/1200px-Golden_Retriever_9-year_old.jpg", "dog_02.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/7/7a/Huskiesatrest.jpg/1200px-Huskiesatrest.jpg", "dog_03.jpg"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/e/e3/Coat_of_arms_of_the_United_Kingdom.svg/1200px-Coat_of_arms_of_the_United_Kingdom.svg.png", "graphic_01.png"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/1200px-PNG_transparency_demonstration_1.png", "graphic_02.png"),
]

# Download images
downloaded_images = []
for url, filename in image_urls:
    try:
        filepath = f"sample_images/{filename}"
        if not os.path.exists(filepath):
            urllib.request.urlretrieve(url, filepath)
        downloaded_images.append(filepath)
        print(f"? Downloaded: {filename}")
    except Exception as e:
        print(f" Failed to download {filename}: {e}")

print(f"\nTotal images available: {len(downloaded_images)}")

# Also create some synthetic images with different properties
from PIL import ImageDraw, ImageFont

def create_synthetic_images():
    """Create synthetic images with known properties for testing"""
    synthetic_dir = "sample_images/synthetic"
    os.makedirs(synthetic_dir, exist_ok=True)
    
    # High resolution image
    img_high_res = Image.new('RGB', (1920, 1080), color='red')
    ImageDraw.Draw(img_high_res).text((100, 500), "High Resolution 1920x1080", fill='white')
    img_high_res.save(f"{synthetic_dir}/high_res.jpg")
    
    # Low resolution image
    img_low_res = Image.new('RGB', (100, 100), color='blue')
    ImageDraw.Draw(img_low_res).text((10, 40), "Low Res", fill='white')
    img_low_res.save(f"{synthetic_dir}/low_res.jpg")
    
    # Grayscale image
    img_gray = Image.new('L', (400, 400), color=128)
    img_gray.save(f"{synthetic_dir}/grayscale.jpg")
    
    # Transparent PNG
    img_rgba = Image.new('RGBA', (500, 500), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img_rgba)
    draw.ellipse([100, 100, 400, 400], fill=(255, 0, 0, 128))
    img_rgba.save(f"{synthetic_dir}/transparent.png")
    
    return [f"{synthetic_dir}/high_res.jpg", f"{synthetic_dir}/low_res.jpg", 
            f"{synthetic_dir}/grayscale.jpg", f"{synthetic_dir}/transparent.png"]

synthetic_images = create_synthetic_images()
all_images = downloaded_images + synthetic_images

print(f"? Created {len(synthetic_images)} synthetic test images")
print(f"Total dataset size: {len(all_images)} images")

In [ ]:
# Load and inspect first image
sample_image_path = all_images[0]
img = Image.open(sample_image_path)

print("=" * 60)
print("FIRST IMAGE INSPECTION")
print("=" * 60)
print(f"File: {sample_image_path}")
print(f"Size (width x height): {img.size}")
print(f"Mode (color space): {img.mode}")
print(f"Format: {img.format}")
print(f"Format Description: {img.format_description if hasattr(img, 'format_description') else 'N/A'}")

# Display the image
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.title(f"Sample Image: {os.path.basename(sample_image_path)}", fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("\n? Image loaded successfully with PIL!")

### Insight: PIL Image Object

The PIL Image object contains rich metadata:
- **Size**: Tuple of (width, height) in pixels
- **Mode**: Color space ('RGB' for color, 'L' for grayscale, 'RGBA' for transparent)
- **Format**: File format detected from header (JPEG, PNG, etc.)
- **Format Description**: Human-readable format information

This metadata is crucial for building preprocessing pipelines that handle different image types correctly!

## 3. Basic Image Properties  

Let's systematically extract properties from all images in our dataset.

In [ ]:
# Extract properties from all images
def extract_image_properties(image_path):
    """Extract comprehensive properties from an image file"""
    try:
        with Image.open(image_path) as img:
            properties = {
                'filename': os.path.basename(image_path),
                'filepath': image_path,
                'width': img.width,
                'height': img.height,
                'mode': img.mode,
                'format': img.format,
                'file_size_kb': os.path.getsize(image_path) / 1024,
                'aspect_ratio': img.width / img.height,
                'megapixels': (img.width * img.height) / 1_000_000,
                'is_rgb': img.mode == 'RGB',
                'is_rgba': img.mode == 'RGBA',
                'is_grayscale': img.mode in ('L', 'LA'),
                'has_transparency': img.mode in ('RGBA', 'LA', 'P')
            }
            
            # Try to get EXIF data if available
            try:
                exif = img._getexif()
                if exif:
                    properties['has_exif'] = True
                    properties['exif_count'] = len(exif)
                else:
                    properties['has_exif'] = False
                    properties['exif_count'] = 0
            except:
                properties['has_exif'] = False
                properties['exif_count'] = 0
                
            return properties
    except Exception as e:
        return {
            'filename': os.path.basename(image_path),
            'filepath': image_path,
            'error': str(e)
        }

# Extract properties for all images
print("=" * 60)
print("EXTRACTING PROPERTIES FROM ALL IMAGES")
print("=" * 60)

image_data = []
for img_path in all_images:
    props = extract_image_properties(img_path)
    image_data.append(props)
    if 'error' not in props:
        print(f"? {props['filename']}: {props['width']}x{props['height']} ({props['mode']}) - {props['file_size_kb']:.1f} KB")
    else:
        print(f"? {props['filename']}: ERROR - {props['error']}")

# Create DataFrame
df_images = pd.DataFrame([d for d in image_data if 'error' not in d])

print(f"\nDataset Summary:")
print(f"   Total images processed: {len(df_images)}")
print(f"   Unique formats: {df_images['format'].unique()}")
print(f"   Color modes: {df_images['mode'].unique()}")
print(f"   Avg file size: {df_images['file_size_kb'].mean():.1f} KB")
print(f"   Resolution range: {df_images['width'].min()}x{df_images['height'].min()} to {df_images['width'].max()}x{df_images['height'].max()}")

In [ ]:
# Visualize image properties
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Image Properties Overview', fontsize=18, fontweight='bold', y=1.02)

# Format distribution
format_counts = df_images['format'].value_counts()
axes[0, 0].pie(format_counts.values, labels=format_counts.index, autopct='%1.1f%%', 
               colors=['#ff6b6b', '#4ecdc4', '#45b7d1'], startangle=90)
axes[0, 0].set_title('Image Format Distribution', fontweight='bold', fontsize=14)

# Color mode distribution
mode_counts = df_images['mode'].value_counts()
bars = axes[0, 1].bar(mode_counts.index, mode_counts.values, 
                       color=['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4'], edgecolor='black')
axes[0, 1].set_title('Color Mode Distribution', fontweight='bold', fontsize=14)
axes[0, 1].set_ylabel('Count')
for bar in bars:
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 0.05,
                    f'{int(height)}', ha='center', va='bottom', fontweight='bold')

# File size distribution
axes[1, 0].hist(df_images['file_size_kb'], bins=15, color='skyblue', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('File Size Distribution (KB)', fontweight='bold', fontsize=14)
axes[1, 0].set_xlabel('File Size (KB)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(df_images['file_size_kb'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {df_images["file_size_kb"].mean():.1f} KB')
axes[1, 0].legend()

# Megapixels distribution
axes[1, 1].hist(df_images['megapixels'], bins=15, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Resolution Distribution (Megapixels)', fontweight='bold', fontsize=14)
axes[1, 1].set_xlabel('Megapixels')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(df_images['megapixels'].mean(), color='red', linestyle='--',
                   label=f'Mean: {df_images["megapixels"].mean():.2f} MP')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n? Basic properties analysis complete!")

### Insight: Dataset Diversity

Our dataset shows good diversity in formats (JPEG, PNG) and color modes (RGB, RGBA, Grayscale). File sizes vary significantly, indicating different compression levels and resolutions. This variety is realistic for production datasets where images come from multiple sources with inconsistent specifications. Understanding this distribution helps design robust preprocessing pipelines!

## 4. Visualizing Individual Images and Creating Grids  

Let's create beautiful visualizations showing multiple images in organized grids.

In [ ]:
# Create image grid visualization
def create_image_grid(image_paths, titles=None, cols=3, figsize=(15, 10)):
    """Create a grid of images with optional titles"""
    n_images = len(image_paths)
    rows = (n_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1:
        axes = axes.reshape(1, -1)
    elif cols == 1:
        axes = axes.reshape(-1, 1)
    
    for idx, img_path in enumerate(image_paths):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col] if rows > 1 else axes[col]
        
        try:
            img = Image.open(img_path)
            ax.imshow(img)
            title = titles[idx] if titles else os.path.basename(img_path)
            ax.set_title(title, fontsize=10, fontweight='bold')
            ax.axis('off')
        except Exception as e:
            ax.text(0.5, 0.5, f'Error loading\n{os.path.basename(img_path)}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Error: {os.path.basename(img_path)}', fontsize=8)
    
    # Hide empty subplots
    for idx in range(n_images, rows * cols):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col] if rows > 1 else axes[col]
        ax.axis('off')
    
    plt.tight_layout()
    return fig

# Create grid of all images
print("=" * 60)
print("?IMAGE GRID VISUALIZATION")
print("=" * 60)

# Create titles with properties
titles = []
for _, row in df_images.iterrows():
    title = f"{row['filename']}\n{row['width']}x{row['height']} | {row['mode']}"
    titles.append(title)

fig = create_image_grid(df_images['filepath'].tolist(), titles=titles, cols=3, figsize=(16, 12))
fig.suptitle('Complete Dataset Overview', fontsize=16, fontweight='bold', y=1.02)
plt.show()

print("\n? Image grid created successfully!")

In [ ]:
# Advanced grid with color mode grouping
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Images Grouped by Color Mode', fontsize=18, fontweight='bold', y=1.02)

# Group by color mode
rgb_images = df_images[df_images['mode'] == 'RGB']['filepath'].tolist()[:4]
rgba_images = df_images[df_images['mode'] == 'RGBA']['filepath'].tolist()[:4]
gray_images = df_images[df_images['mode'] == 'L']['filepath'].tolist()[:4]
other_images = df_images[~df_images['mode'].isin(['RGB', 'RGBA', 'L'])]['filepath'].tolist()[:4]

modes_data = [
    (rgb_images, 'RGB (True Color)', 0, 0),
    (rgba_images, 'RGBA (With Transparency)', 0, 1),
    (gray_images, 'Grayscale (L)', 1, 0),
    (other_images, 'Other Modes', 1, 1)
]

for img_list, title, row, col in modes_data:
    ax = axes[row, col]
    if img_list:
        # Create mini-grid within each subplot
        sub_cols = min(2, len(img_list))
        sub_rows = (len(img_list) + sub_cols - 1) // sub_cols
        
        for idx, img_path in enumerate(img_list[:4]):  # Max 4 per category
            try:
                img = Image.open(img_path)
                # Calculate position for mini-image
                sub_col = idx % sub_cols
                sub_row = idx // sub_cols
                # Create inset axes
                inset_width = 0.45
                inset_height = 0.45
                x_pos = 0.05 + sub_col * (inset_width + 0.05)
                y_pos = 0.5 - sub_row * (inset_height + 0.05) if sub_rows > 1 else 0.25
                
                inset_ax = ax.inset_axes([x_pos, y_pos, inset_width, inset_height])
                inset_ax.imshow(img)
                inset_ax.axis('off')
                inset_ax.set_title(os.path.basename(img_path)[:15], fontsize=8)
            except:
                pass
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    ax.axis('off')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("\n? Color mode grouping visualization complete!")

### Insight: Visual Organization

Grouping images by color mode reveals important preprocessing requirements:
- **RGB images** are standard for most CNN architectures
- **RGBA images** need alpha channel handling (either drop or use as mask)
- **Grayscale images** can be replicated to 3 channels or processed with specialized models
- Mixed modes require conversion strategies before batch processing!

## 5. Image Statistics Analysis  

Now let's dive into pixel-level statistics using PIL's ImageStat module.

In [ ]:
# Calculate pixel statistics for each image
def calculate_image_statistics(image_path):
    """Calculate detailed pixel statistics using PIL"""
    try:
        with Image.open(image_path) as img:
            # Convert to RGB for consistent analysis
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            # Get statistics using ImageStat
            stat = ImageStat.Stat(img)
            
            # Convert to numpy for additional calculations
            img_array = np.array(img)
            
            stats = {
                'filename': os.path.basename(image_path),
                'filepath': image_path,
                
                # Channel means (R, G, B)
                'mean_r': stat.mean[0],
                'mean_g': stat.mean[1],
                'mean_b': stat.mean[2],
                'mean_rgb': np.mean(stat.mean),
                
                # Channel std (R, G, B)
                'std_r': stat.stddev[0],
                'std_g': stat.stddev[1],
                'std_b': stat.stddev[2],
                'mean_std': np.mean(stat.stddev),
                
                # Min/Max values
                'min_r': stat.extrema[0][0],
                'max_r': stat.extrema[0][1],
                'min_g': stat.extrema[1][0],
                'max_g': stat.extrema[1][1],
                'min_b': stat.extrema[2][0],
                'max_b': stat.extrema[2][1],
                
                # Brightness (RMS)
                'rms': stat.rms[0] if hasattr(stat, 'rms') else np.sqrt(np.mean(img_array**2)),
                
                # Additional numpy-based stats
                'median_r': np.median(img_array[:,:,0]),
                'median_g': np.median(img_array[:,:,1]),
                'median_b': np.median(img_array[:,:,2]),
                
                # Contrast (difference between max and min)
                'contrast_r': stat.extrema[0][1] - stat.extrema[0][0],
                'contrast_g': stat.extrema[1][1] - stat.extrema[1][0],
                'contrast_b': stat.extrema[2][1] - stat.extrema[2][0],
                'mean_contrast': np.mean([
                    stat.extrema[0][1] - stat.extrema[0][0],
                    stat.extrema[1][1] - stat.extrema[1][0],
                    stat.extrema[2][1] - stat.extrema[2][0]
                ])
            }
            
            return stats
    except Exception as e:
        return {
            'filename': os.path.basename(image_path),
            'filepath': image_path,
            'error': str(e)
        }

print("=" * 60)
print("CALCULATING PIXEL STATISTICS")
print("=" * 60)

stats_data = []
for img_path in all_images:
    stats = calculate_image_statistics(img_path)
    if 'error' not in stats:
        stats_data.append(stats)
        print(f"? {stats['filename']}: Mean={stats['mean_rgb']:.1f}, Std={stats['mean_std']:.1f}, Contrast={stats['mean_contrast']:.1f}")
    else:
        print(f"? {stats['filename']}: ERROR")

df_stats = pd.DataFrame(stats_data)

print(f"\nStatistics Summary:")
print(f"   Images processed: {len(df_stats)}")
print(f"   Avg brightness (mean): {df_stats['mean_rgb'].mean():.1f} (0-255 scale)")
print(f"   Avg contrast: {df_stats['mean_contrast'].mean():.1f}")
print(f"   Brightness range: {df_stats['mean_rgb'].min():.1f} to {df_stats['mean_rgb'].max():.1f}")

In [ ]:
# Visualize pixel statistics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Pixel Statistics Analysis', fontsize=18, fontweight='bold', y=1.02)

# Mean brightness distribution
axes[0, 0].hist(df_stats['mean_rgb'], bins=15, color='gold', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Mean Brightness Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Mean Pixel Value (0-255)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].axvline(128, color='red', linestyle='--', label='Midpoint (128)')
axes[0, 0].legend()

# Standard deviation (variability)
axes[0, 1].hist(df_stats['mean_std'], bins=15, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Pixel Variability (Std Dev)', fontweight='bold')
axes[0, 1].set_xlabel('Mean Std Dev (0-255)')
axes[0, 1].set_ylabel('Count')

# Contrast distribution
axes[0, 2].hist(df_stats['mean_contrast'], bins=15, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 2].set_title('Contrast Distribution', fontweight='bold')
axes[0, 2].set_xlabel('Contrast (Max - Min)')
axes[0, 2].set_ylabel('Count')

# RGB Mean comparison
x_pos = np.arange(len(df_stats))
width = 0.25
axes[1, 0].bar(x_pos - width, df_stats['mean_r'], width, label='Red', color='red', alpha=0.7)
axes[1, 0].bar(x_pos, df_stats['mean_g'], width, label='Green', color='green', alpha=0.7)
axes[1, 0].bar(x_pos + width, df_stats['mean_b'], width, label='Blue', color='blue', alpha=0.7)
axes[1, 0].set_title('Mean RGB Values by Image', fontweight='bold')
axes[1, 0].set_xlabel('Image Index')
axes[1, 0].set_ylabel('Mean Pixel Value')
axes[1, 0].legend()
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels([f"{i+1}" for i in range(len(df_stats))], rotation=45)

# Brightness vs Contrast scatter
scatter = axes[1, 1].scatter(df_stats['mean_rgb'], df_stats['mean_contrast'], 
                           c=df_stats['mean_std'], cmap='viridis', s=100, edgecolors='black')
axes[1, 1].set_title('Brightness vs Contrast', fontweight='bold')
axes[1, 1].set_xlabel('Mean Brightness')
axes[1, 1].set_ylabel('Contrast')
plt.colorbar(scatter, ax=axes[1, 1], label='Std Dev (Variability)')

# RMS distribution
axes[1, 2].hist(df_stats['rms'], bins=15, color='mediumpurple', edgecolor='black', alpha=0.7)
axes[1, 2].set_title('RMS (Root Mean Square)', fontweight='bold')
axes[1, 2].set_xlabel('RMS Value')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("\n? Pixel statistics visualization complete!")

### Insight: Pixel Statistics Patterns

Pixel statistics reveal important image characteristics:
- **Mean brightness** around 128 indicates balanced exposure; extremes suggest over/under-exposure
- **High standard deviation** indicates rich texture and detail (good for training)
- **Low contrast** images may be blurry, foggy, or low-quality
- **RMS values** help identify uniform vs. textured images
- RGB channel balance reveals color casts (e.g., red-tinted or blue-tinted images)

Images with extreme statistics may need normalization or could be outliers!

## 6. Color Analysis  

Let's perform deep color channel analysis including histograms.

In [ ]:
# Detailed color channel analysis
def analyze_color_channels(image_path):
    """Analyze individual color channels"""
    with Image.open(image_path) as img:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        img_array = np.array(img)
        
        # Split channels
        r_channel = img_array[:,:,0]
        g_channel = img_array[:,:,1]
        b_channel = img_array[:,:,2]
        
        # Calculate histograms
        r_hist, _ = np.histogram(r_channel.flatten(), bins=256, range=(0, 256))
        g_hist, _ = np.histogram(g_channel.flatten(), bins=256, range=(0, 256))
        b_hist, _ = np.histogram(b_channel.flatten(), bins=256, range=(0, 256))
        
        return {
            'filename': os.path.basename(image_path),
            'r_hist': r_hist,
            'g_hist': g_hist,
            'b_hist': b_hist,
            'r_mean': r_channel.mean(),
            'g_mean': g_channel.mean(),
            'b_mean': b_channel.mean(),
            'dominant_color': np.argmax([r_hist.sum(), g_hist.sum(), b_hist.sum()])  # 0=R, 1=G, 2=B
        }

# Analyze first 4 images
print("=" * 60)
print("COLOR CHANNEL ANALYSIS")
print("=" * 60)

sample_paths = df_images['filepath'].tolist()[:4]
color_data = [analyze_color_channels(path) for path in sample_paths]

# Create comprehensive color visualization
fig, axes = plt.subplots(4, 4, figsize=(20, 16))
fig.suptitle('RGB Channel Histograms', fontsize=18, fontweight='bold', y=1.02)

colors = ['red', 'green', 'blue']
channel_names = ['Red', 'Green', 'Blue']

for idx, data in enumerate(color_data):
    # Show image
    img = Image.open(data['filename'] if os.path.exists(data['filename']) else sample_paths[idx])
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f"{data['filename']}\nDominant: {channel_names[data['dominant_color']]}", 
                          fontsize=10, fontweight='bold')
    axes[idx, 0].axis('off')
    
    # Plot histograms
    for i, (hist, color, name) in enumerate(zip([data['r_hist'], data['g_hist'], data['b_hist']], 
                                                  colors, channel_names)):
        ax = axes[idx, i+1] if i < 2 else axes[idx, i+1] if i == 2 else None
        if i == 2:
            ax = axes[idx, 2]
        ax.plot(hist, color=color, linewidth=2, label=f'{name} (?={data[('r_mean' if name == 'Red' else 'g_mean' if name == 'Green' else 'b_mean')]:.1f})')
        ax.set_xlim(0, 255)
        ax.set_title(f'{name} Channel Histogram', fontsize=10)
        ax.set_xlabel('Pixel Value')
        ax.set_ylabel('Frequency')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n? Color channel analysis complete!")

In [ ]:
# Average color analysis across dataset
print("=" * 60)
print("DATASET-WIDE COLOR ANALYSIS")
print("=" * 60)

# Calculate average colors
avg_colors = []
for _, row in df_stats.iterrows():
    avg_colors.append([row['mean_r']/255, row['mean_g']/255, row['mean_b']/255])

avg_colors = np.array(avg_colors)

# Create color distribution visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Dataset Color Distribution', fontsize=18, fontweight='bold', y=1.02)

# RGB 3D scatter (simulated as 2D projections)
axes[0, 0].scatter(df_stats['mean_r'], df_stats['mean_g'], 
                  c=avg_colors, s=200, edgecolors='black', linewidth=1)
axes[0, 0].set_xlabel('Red Channel Mean')
axes[0, 0].set_ylabel('Green Channel Mean')
axes[0, 0].set_title('Red vs Green Channels', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(df_stats['mean_r'], df_stats['mean_b'], 
                  c=avg_colors, s=200, edgecolors='black', linewidth=1)
axes[0, 1].set_xlabel('Red Channel Mean')
axes[0, 1].set_ylabel('Blue Channel Mean')
axes[0, 1].set_title('Red vs Blue Channels', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Color bar chart
x_pos = np.arange(len(df_stats))
width = 0.25
axes[1, 0].bar(x_pos - width, df_stats['mean_r'], width, label='Red', color='red', alpha=0.8)
axes[1, 0].bar(x_pos, df_stats['mean_g'], width, label='Green', color='green', alpha=0.8)
axes[1, 0].bar(x_pos + width, df_stats['mean_b'], width, label='Blue', color='blue', alpha=0.8)
axes[1, 0].set_xlabel('Image Index')
axes[1, 0].set_ylabel('Mean Channel Value')
axes[1, 0].set_title('RGB Channel Comparison', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels([f"{i+1}" for i in range(len(df_stats))])

# Color palette visualization
axes[1, 1].barh(range(len(df_stats)), [1]*len(df_stats), 
               color=avg_colors, edgecolor='black', linewidth=1)
axes[1, 1].set_yticks(range(len(df_stats)))
axes[1, 1].set_yticklabels([f"Img {i+1}" for i in range(len(df_stats))])
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_title('Average Color Palette', fontweight='bold')
axes[1, 1].set_xlabel('Color Representation')
axes[1, 1].set_xticks([])

plt.tight_layout()
plt.show()

# Print color statistics
print(f"\nColor Statistics:")
print(f"   Red dominance: {(df_stats['mean_r'] > df_stats[['mean_g', 'mean_b']].max(axis=1)).sum()} images")
print(f"   Green dominance: {(df_stats['mean_g'] > df_stats[['mean_r', 'mean_b']].max(axis=1)).sum()} images")
print(f"   Blue dominance: {(df_stats['mean_b'] > df_stats[['mean_r', 'mean_g']].max(axis=1)).sum()} images")
print(f"   Balanced (all channels within 20): {(df_stats[['mean_r', 'mean_g', 'mean_b']].max(axis=1) - df_stats[['mean_r', 'mean_g', 'mean_b']].min(axis=1) < 20).sum()} images")

### Insight: Color Distribution Patterns

Color analysis reveals dataset biases:
- **Dominant red** suggests sunset, indoor, or warm-temperature images
- **Dominant green** indicates outdoor, nature, or vegetation-heavy images  
- **Dominant blue** implies sky, water, or cool-temperature images
- **Balanced colors** suggest neutral lighting or post-processing

Color imbalance in the dataset can lead to model bias if all "cat" images are warm-toned and "dog" images cool-toned, the model might learn color associations rather than semantic features!

## 7. Image Quality and Issues Detection  

Let's identify potential quality issues like blur, noise, and corruption.

In [ ]:
# Image quality assessment functions
def detect_blur(image_path, threshold=100):
    """Detect blur using Laplacian variance"""
    try:
        with Image.open(image_path) as img:
            if img.mode != 'L':
                img = img.convert('L')
            # Apply Laplacian filter
            laplacian = img.filter(ImageFilter.FIND_EDGES)
            variance = np.var(np.array(laplacian))
            return variance < threshold, variance
    except:
        return None, 0

def detect_corruption(image_path):
    """Check if image file is corrupted"""
    try:
        with Image.open(image_path) as img:
            img.verify()  # Verify without loading
        with Image.open(image_path) as img:
            img.load()    # Actually load pixels
        return False, "OK"
    except Exception as e:
        return True, str(e)

def check_duplicates(image_paths):
    """Check for potential duplicate images using simple hashing"""
    hashes = {}
    duplicates = []
    
    for path in image_paths:
        try:
            with Image.open(path) as img:
                # Simple hash based on size and mode
                img_hash = (img.size, img.mode, os.path.getsize(path))
                if img_hash in hashes:
                    duplicates.append((path, hashes[img_hash]))
                else:
                    hashes[img_hash] = path
        except:
            pass
    
    return duplicates

print("=" * 60)
print("IMAGE QUALITY ASSESSMENT")
print("=" * 60)

# Check for corruption
print("\n1. CHECKING FOR CORRUPTION:")
corrupted = []
for img_path in all_images:
    is_corrupt, msg = detect_corruption(img_path)
    if is_corrupt:
        corrupted.append((img_path, msg))
        print(f"   {os.path.basename(img_path)}: {msg[:50]}")
    else:
        print(f"   {os.path.basename(img_path)}: OK")

# Check for blur
print(f"\n2. CHECKING FOR BLUR (threshold=100):")
blurry_images = []
blur_scores = []
for img_path in all_images:
    is_blurry, variance = detect_blur(img_path)
    blur_scores.append(variance)
    if is_blurry is True:
        blurry_images.append((img_path, variance))
        print(f"    {os.path.basename(img_path)}: Blurry (variance={variance:.1f})")
    elif is_blurry is False:
        print(f"   {os.path.basename(img_path)}: Sharp (variance={variance:.1f})")
    else:
        print(f"   {os.path.basename(img_path)}: Could not assess")

# Check for duplicates
print(f"\n3. CHECKING FOR DUPLICATES:")
duplicates = check_duplicates(all_images)
if duplicates:
    for dup, orig in duplicates:
        print(f"    Potential duplicate: {os.path.basename(dup)} {os.path.basename(orig)}")
else:
    print(f"   No obvious duplicates found")

print(f"\nQuality Summary:")
print(f"   Corrupted: {len(corrupted)}/{len(all_images)}")
print(f"   Blurry: {len(blurry_images)}/{len(all_images)}")
print(f"   Duplicates: {len(duplicates)}/{len(all_images)}")
print(f"   Avg blur variance: {np.mean([s for s in blur_scores if s > 0]):.1f}")

In [ ]:
# Visualize quality metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Image Quality Analysis', fontsize=18, fontweight='bold', y=1.02)

# Blur variance distribution
valid_blur_scores = [s for s in blur_scores if s > 0]
axes[0, 0].hist(valid_blur_scores, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(100, color='red', linestyle='--', linewidth=2, label='Blur Threshold')
axes[0, 0].set_title('Blur Variance Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Laplacian Variance (higher = sharper)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# Blur vs file size
file_sizes = [os.path.getsize(p)/1024 for p in all_images if os.path.exists(p)]
if len(file_sizes) == len(valid_blur_scores):
    axes[0, 1].scatter(file_sizes, valid_blur_scores, s=100, alpha=0.7, edgecolors='black')
    axes[0, 1].set_xlabel('File Size (KB)')
    axes[0, 1].set_ylabel('Blur Variance')
    axes[0, 1].set_title('File Size vs Sharpness', fontweight='bold')
    axes[0, 1].axhline(100, color='red', linestyle='--', alpha=0.5, label='Blur Threshold')
    axes[0, 1].legend()

# Quality categories
quality_categories = ['Corrupted', 'Blurry', 'Low Contrast', 'Good Quality']
quality_counts = [
    len(corrupted),
    len(blurry_images),
    len(df_stats[df_stats['mean_contrast'] < 50]),
    len(all_images) - len(corrupted) - len(blurry_images) - len(df_stats[df_stats['mean_contrast'] < 50])
]
colors = ['#ff6b6b', '#feca57', '#ff9ff3', '#54a0ff']
axes[1, 0].pie(quality_counts, labels=quality_categories, autopct='%1.1f%%', 
               colors=colors, startangle=90, explode=(0.05, 0.05, 0.05, 0))
axes[1, 0].set_title('Image Quality Distribution', fontweight='bold')

# Contrast vs Brightness (quality indicators)
scatter = axes[1, 1].scatter(df_stats['mean_rgb'], df_stats['mean_contrast'], 
                            c=valid_blur_scores[:len(df_stats)], cmap='RdYlGn', 
                            s=150, edgecolors='black', linewidth=1)
axes[1, 1].set_xlabel('Mean Brightness')
axes[1, 1].set_ylabel('Contrast')
axes[1, 1].set_title('Quality Matrix (color=sharpness)', fontweight='bold')
plt.colorbar(scatter, ax=axes[1, 1], label='Blur Variance')

# Add quadrant lines
axes[1, 1].axhline(50, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].axvline(128, color='gray', linestyle='--', alpha=0.5)
axes[1, 1].text(200, 200, 'Good\nQuality', ha='center', va='center', 
                bbox=dict(boxstyle='round', facecolor='green', alpha=0.3))
axes[1, 1].text(50, 200, 'Dark but\nHigh Contrast', ha='center', va='center',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

plt.tight_layout()
plt.show()

print("\n? Quality assessment visualization complete!")

### Insight: Quality Control Strategy

Quality assessment reveals:
- **Blurry images** (low variance) may need removal or sharpening preprocessing
- **Corrupted files** must be removed before training to prevent crashes
- **Duplicates** waste training resources and can cause overfitting
- **Low contrast** images might benefit from histogram equalization

Implementing automated quality checks in your data pipeline prevents these issues from affecting model performance!

## 8. Aspect Ratio and Resolution Analysis  

Understanding the distribution of image dimensions is crucial for input sizing decisions.

In [ ]:
# Aspect ratio and resolution analysis
print("=" * 60)
print("ASPECT RATIO & RESOLUTION ANALYSIS")
print("=" * 60)

# Calculate aspect ratio categories
def categorize_aspect_ratio(ratio):
    if ratio < 0.8:
        return 'Portrait/Tall'
    elif ratio > 1.2:
        return 'Landscape/Wide'
    else:
        return 'Square/Near-Square'

def categorize_resolution(mp):
    if mp < 0.1:
        return 'Thumbnail (<0.1MP)'
    elif mp < 1:
        return 'Low (0.1-1MP)'
    elif mp < 5:
        return 'Medium (1-5MP)'
    elif mp < 10:
        return 'High (5-10MP)'
    else:
        return 'Very High (>10MP)'

df_images['aspect_category'] = df_images['aspect_ratio'].apply(categorize_aspect_ratio)
df_images['resolution_category'] = df_images['megapixels'].apply(categorize_resolution)

print("\nASPECT RATIO DISTRIBUTION:")
aspect_counts = df_images['aspect_category'].value_counts()
for cat, count in aspect_counts.items():
    print(f"   {cat}: {count} ({count/len(df_images)*100:.1f}%)")

print(f"\nRESOLUTION DISTRIBUTION:")
res_counts = df_images['resolution_category'].value_counts()
for cat, count in res_counts.items():
    print(f"   {cat}: {count} ({count/len(df_images)*100:.1f}%)")

print(f"\nASPECT RATIO STATISTICS:")
print(f"   Min ratio: {df_images['aspect_ratio'].min():.3f}")
print(f"   Max ratio: {df_images['aspect_ratio'].max():.3f}")
print(f"   Mean ratio: {df_images['aspect_ratio'].mean():.3f}")
print(f"   Std dev: {df_images['aspect_ratio'].std():.3f}")

# Common resolutions
df_images['resolution_str'] = df_images['width'].astype(str) + 'x' + df_images['height'].astype(str)
common_resolutions = df_images['resolution_str'].value_counts().head(5)
print(f"\nTOP 5 RESOLUTIONS:")
for res, count in common_resolutions.items():
    print(f"   {res}: {count} images")

In [ ]:
# Visualize aspect ratio and resolution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Aspect Ratio & Resolution Analysis', fontsize=18, fontweight='bold', y=1.02)

# Aspect ratio distribution
aspect_counts.plot(kind='bar', ax=axes[0, 0], color=['#ff6b6b', '#4ecdc4', '#45b7d1'], edgecolor='black')
axes[0, 0].set_title('Aspect Ratio Categories', fontweight='bold')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=45)
for i, v in enumerate(aspect_counts.values):
    axes[0, 0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Resolution categories
res_counts.plot(kind='bar', ax=axes[0, 1], color='mediumpurple', edgecolor='black')
axes[0, 1].set_title('Resolution Categories', fontweight='bold')
axes[0, 1].set_ylabel('Count')
axes[0, 1].tick_params(axis='x', rotation=45)

# Aspect ratio scatter (width vs height)
colors_map = {'Portrait/Tall': '#ff6b6b', 'Square/Near-Square': '#feca57', 'Landscape/Wide': '#4ecdc4'}
for category in df_images['aspect_category'].unique():
    mask = df_images['aspect_category'] == category
    axes[1, 0].scatter(df_images[mask]['width'], df_images[mask]['height'], 
                      label=category, s=150, alpha=0.7, 
                      color=colors_map.get(category, 'gray'), edgecolors='black')
axes[1, 0].set_xlabel('Width (pixels)')
axes[1, 0].set_ylabel('Height (pixels)')
axes[1, 0].set_title('Resolution Scatter Plot', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(axes[1, 0].get_xlim()[1], color='gray', linestyle='--', alpha=0.5)

# Aspect ratio histogram with vertical lines
axes[1, 1].hist(df_images['aspect_ratio'], bins=15, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(1.0, color='red', linestyle='--', linewidth=2, label='Square (1.0)')
axes[1, 1].axvline(4/3, color='blue', linestyle='--', linewidth=2, label='4:3 (1.33)')
axes[1, 1].axvline(16/9, color='green', linestyle='--', linewidth=2, label='16:9 (1.78)')
axes[1, 1].set_xlabel('Aspect Ratio (width/height)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Aspect Ratio Distribution', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n? Aspect ratio and resolution analysis complete!")

### Insight: Dimension Strategy

Aspect ratio analysis informs preprocessing decisions:
- **Mixed ratios** require resizing strategies (stretch, crop, or pad)
- **Standard ratios** (4:3, 16:9) suggest photography/camera sources
- **Square images** are easiest for CNNs (common input size like 224x224)
- **Extreme ratios** (very tall or wide) may need special handling or exclusion

For deep learning, consider:
- **Resize** to square for simplicity (may distort content)
- **Center crop** to square (may lose important edge content)
- **Padding** to square (preserves aspect ratio but adds empty space)
- **Multiple scales** for multi-scale training

## 9. Batch Image Analysis  

Let's create efficient batch processing functions for analyzing entire folders.

In [ ]:
# Batch processing functions
class ImageDatasetAnalyzer:
    """Class for batch analysis of image datasets"""
    
    def __init__(self, folder_path):
        self.folder_path = Path(folder_path)
        self.supported_formats = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
        self.image_files = []
        self.results = []
        
    def scan_folder(self):
        """Scan folder for image files"""
        self.image_files = []
        for ext in self.supported_formats:
            self.image_files.extend(self.folder_path.glob(f'**/*{ext}'))
            self.image_files.extend(self.folder_path.glob(f'**/*{ext.upper()}'))
        self.image_files = list(set(self.image_files))  # Remove duplicates
        print(f"Found {len(self.image_files)} images in {self.folder_path}")
        return self
    
    def analyze_batch(self, max_images=None):
        """Analyze all images in batch"""
        images_to_process = self.image_files[:max_images] if max_images else self.image_files
        
        print(f"Analyzing {len(images_to_process)} images...")
        
        for idx, img_path in enumerate(images_to_process):
            if idx % 10 == 0:
                print(f"   Progress: {idx}/{len(images_to_process)}")
            
            try:
                with Image.open(img_path) as img:
                    # Basic properties
                    result = {
                        'filename': img_path.name,
                        'filepath': str(img_path),
                        'width': img.width,
                        'height': img.height,
                        'mode': img.mode,
                        'format': img.format,
                        'file_size_kb': img_path.stat().st_size / 1024,
                        'aspect_ratio': img.width / img.height,
                        'megapixels': (img.width * img.height) / 1_000_000
                    }
                    
                    # Pixel statistics if RGB
                    if img.mode == 'RGB':
                        stat = ImageStat.Stat(img)
                        result.update({
                            'mean_r': stat.mean[0],
                            'mean_g': stat.mean[1],
                            'mean_b': stat.mean[2],
                            'std_r': stat.stddev[0],
                            'std_g': stat.stddev[1],
                            'std_b': stat.stddev[2],
                        })
                    
                    self.results.append(result)
                    
            except Exception as e:
                self.results.append({
                    'filename': img_path.name,
                    'filepath': str(img_path),
                    'error': str(e)
                })
        
        return self
    
    def get_dataframe(self):
        """Return results as DataFrame"""
        return pd.DataFrame(self.results)
    
    def generate_report(self):
        """Generate summary report"""
        df = self.get_dataframe()
        if 'error' in df.columns:
            valid_df = df[df['error'].isna()].copy()
        else:
            valid_df = df.copy()
        
        report = {
            'total_images': len(df),
            'successful': len(valid_df),
            'failed': len(df) - len(valid_df),
            'formats': valid_df['format'].value_counts().to_dict() if 'format' in valid_df.columns else {},
            'modes': valid_df['mode'].value_counts().to_dict() if 'mode' in valid_df.columns else {},
            'avg_resolution': (valid_df['width'].mean(), valid_df['height'].mean()) if 'width' in valid_df.columns else (0, 0),
            'total_size_mb': valid_df['file_size_kb'].sum() / 1024 if 'file_size_kb' in valid_df.columns else 0
        }
        
        return report

# Demonstrate batch analyzer
print("=" * 60)
print("BATCH ANALYSIS DEMONSTRATION")
print("=" * 60)

analyzer = ImageDatasetAnalyzer('sample_images')
analyzer.scan_folder()
analyzer.analyze_batch()
batch_df = analyzer.get_dataframe()
report = analyzer.generate_report()

print(f"\nBATCH ANALYSIS REPORT:")
print(f"   Total images: {report['total_images']}")
print(f"   Successfully analyzed: {report['successful']}")
print(f"   Failed: {report['failed']}")
print(f"   Formats: {report['formats']}")
print(f"   Color modes: {report['modes']}")
print(f"   Average resolution: {report['avg_resolution'][0]:.0f} x {report['avg_resolution'][1]:.0f}")
print(f"   Total dataset size: {report['total_size_mb']:.2f} MB")

print(f"\nSample of batch results:")
print(batch_df[['filename', 'width', 'height', 'mode', 'format']].head())

In [ ]:
# Visualize batch analysis results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Batch Dataset Analysis Results', fontsize=18, fontweight='bold', y=1.02)

if 'error' in batch_df.columns:
    valid_df = batch_df[batch_df['error'].isna()].copy()
else:
    valid_df = batch_df.copy()

# Format distribution
if 'format' in valid_df.columns:
    format_counts = valid_df['format'].value_counts()
    axes[0, 0].pie(format_counts.values, labels=format_counts.index, autopct='%1.1f%%',
                   colors=['#ff6b6b', '#4ecdc4', '#45b7d1'], startangle=90)
    axes[0, 0].set_title('Image Formats', fontweight='bold')

# Resolution scatter
if 'width' in valid_df.columns:
    scatter = axes[0, 1].scatter(valid_df['width'], valid_df['height'], 
                                c=valid_df['file_size_kb'] if 'file_size_kb' in valid_df.columns else 'blue',
                                s=100, alpha=0.6, edgecolors='black', cmap='viridis')
    axes[0, 1].set_xlabel('Width (pixels)')
    axes[0, 1].set_ylabel('Height (pixels)')
    axes[0, 1].set_title('Resolution Distribution', fontweight='bold')
    if 'file_size_kb' in valid_df.columns:
        plt.colorbar(scatter, ax=axes[0, 1], label='File Size (KB)')

# File size distribution
if 'file_size_kb' in valid_df.columns:
    axes[1, 0].hist(valid_df['file_size_kb'], bins=20, color='lightblue', edgecolor='black', alpha=0.7)
    axes[1, 0].set_xlabel('File Size (KB)')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('File Size Distribution', fontweight='bold')
    axes[1, 0].axvline(valid_df['file_size_kb'].mean(), color='red', linestyle='--', 
                       label=f'Mean: {valid_df["file_size_kb"].mean():.1f} KB')
    axes[1, 0].legend()

# Aspect ratio distribution
if 'aspect_ratio' in valid_df.columns:
    axes[1, 1].hist(valid_df['aspect_ratio'], bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[1, 1].set_xlabel('Aspect Ratio')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_title('Aspect Ratio Distribution', fontweight='bold')
    axes[1, 1].axvline(1.0, color='red', linestyle='--', alpha=0.7, label='Square')
    axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("\n? Batch analysis visualization complete!")

### Insight: Batch Processing Power

The batch analyzer class provides:
- **Scalability** Handle thousands of images efficiently
- **Error resilience** Continue processing even if some files fail
- **Comprehensive metrics** All important properties in one DataFrame
- **Extensibility** Easy to add custom analysis methods

For production pipelines, consider adding:
- Parallel processing (multiprocessing)
- Progress bars (tqdm)
- Memory-efficient generators for large datasets
- Export to CSV/Parquet for further analysis

## 10. Key Insights and Recommendations  

Let's synthesize our findings into actionable recommendations for computer vision pipelines.

In [ ]:
# Generate comprehensive insights report
print("=" * 80)
print("KEY INSIGHTS FOR COMPUTER VISION PIPELINES")
print("=" * 80)

# Combine all data
all_data = df_images.merge(df_stats, on='filepath', how='left')

insights = {
    "Dataset Overview": {
        "Total Images": f"{len(all_images)}",
        "Unique Formats": f"{df_images['format'].nunique()} ({', '.join(df_images['format'].unique())})",
        "Color Modes": f"{df_images['mode'].nunique()} modes ({', '.join(df_images['mode'].unique())})",
        "Total Size": f"{df_images['file_size_kb'].sum()/1024:.2f} MB",
        "Avg File Size": f"{df_images['file_size_kb'].mean():.1f} KB"
    },
    "Resolution Analysis": {
        "Resolution Range": f"{df_images['width'].min()}x{df_images['height'].min()} to {df_images['width'].max()}x{df_images['height'].max()}",
        "Average Resolution": f"{df_images['width'].mean():.0f} x {df_images['height'].mean():.0f}",
        "Megapixel Range": f"{df_images['megapixels'].min():.2f} - {df_images['megapixels'].max():.2f} MP",
        "Most Common Aspect Ratio": f"{df_images['aspect_category'].mode()[0]}"
    },
    "Color & Quality": {
        "Avg Brightness": f"{df_stats['mean_rgb'].mean():.1f}/255",
        "Brightness Range": f"{df_stats['mean_rgb'].min():.1f} - {df_stats['mean_rgb'].max():.1f}",
        "Avg Contrast": f"{df_stats['mean_contrast'].mean():.1f}",
        "Avg Sharpness (variance)": f"{np.mean([s for s in blur_scores if s > 0]):.1f}",
        "Dominant Color Bias": "Mixed (no strong bias detected)" if df_stats[['mean_r', 'mean_g', 'mean_b']].std(axis=1).mean() > 30 else "Low variance"
    },
    "Quality Issues": {
        "Corrupted Files": f"{len(corrupted)} ({len(corrupted)/len(all_images)*100:.1f}%)",
        "Blurry Images": f"{len(blurry_images)} ({len(blurry_images)/len(all_images)*100:.1f}%)",
        "Low Contrast (<50)": f"{len(df_stats[df_stats['mean_contrast'] < 50])} ({len(df_stats[df_stats['mean_contrast'] < 50])/len(df_stats)*100:.1f}%)",
        "Duplicates Found": f"{len(duplicates)} pairs"
    }
}

for category, items in insights.items():
    print(f"\n{category}:")
    print("-" * 60)
    for key, value in items.items():
        print(f"   {key:<25}: {value}")

# Recommendations
print(f"\n{'=' * 80}")
print("PREPROCESSING RECOMMENDATIONS")
print(f"{'=' * 80}")

recommendations = [
    ("Input Size", f"Resize to {int(df_images['width'].quantile(0.5))}x{int(df_images['height'].quantile(0.5))} or standard 224x224"),
    ("Aspect Ratio", "Use center-crop or padding to maintain aspect ratio, avoid stretching"),
    ("Color Mode", "Convert all to RGB (3 channels) for standard CNN architectures"),
    ("Normalization", f"Use mean=[{df_stats['mean_r'].mean()/255:.3f}, {df_stats['mean_g'].mean()/255:.3f}, {df_stats['mean_b'].mean()/255:.3f}], std=[0.229, 0.224, 0.225]"),
    ("Augmentation", "Random horizontal flip, rotation +/-15 deg, brightness +/-20% (based on dataset variance)"),
    ("Quality Filter", "Remove images with contrast <30 or blur variance <50"),
    ("Batch Size", f"Based on avg size {df_images['file_size_kb'].mean():.0f}KB, use batch size 32-64 for 8GB GPU"),
    ("Validation Split", "Stratify by aspect ratio category to ensure validation set is representative")
]

for i, (aspect, recommendation) in enumerate(recommendations, 1):
    print(f"{i}. {aspect:<15}: {recommendation}")

print(f"\n{'=' * 80}")
print("POTENTIAL ISSUES & MITIGATIONS")
print(f"{'=' * 80}")

issues = [
    ("Mixed Resolutions", "Use torchvision.transforms.Resize or tf.image.resize with consistent method"),
    ("Color Mode Mix", "Add converter: if img.mode != 'RGB': img = img.convert('RGB')"),
    ("Large File Sizes", "Consider pre-processing to JPEG quality 85, or use tf.data for on-the-fly loading"),
    ("Extreme Aspect Ratios", "Flag for review or use letterboxing (padding with black/gray)"),
    ("Corrupted Files", "Implement try-except in data loader; maintain clean file list"),
    ("Class Imbalance", "If applicable, use weighted sampling or oversampling")
]

for issue, mitigation in issues:
    print(f"   {issue:<20}: {mitigation}")

print("\n? Analysis complete!")

In [ ]:
# Final executive dashboard
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(3, 4, hspace=0.3, wspace=0.3)

fig.suptitle('Image EDA Executive Dashboard', fontsize=20, fontweight='bold', y=0.98)

# 1. Dataset summary (top left)
ax1 = fig.add_subplot(gs[0, 0])
metrics = ['Images', 'Formats', 'Modes', 'Size(MB)']
values = [len(all_images), df_images['format'].nunique(), df_images['mode'].nunique(), 
          df_images['file_size_kb'].sum()/1024]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']
bars = ax1.bar(metrics, values, color=colors, edgecolor='black', linewidth=2)
ax1.set_title('Dataset Overview', fontweight='bold', fontsize=12)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01, 
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# 2. Resolution distribution (top middle-left)
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df_images['megapixels'], bins=10, color='mediumpurple', edgecolor='black', alpha=0.7)
ax2.set_title('Resolution (MP)', fontweight='bold', fontsize=12)
ax2.set_xlabel('Megapixels')

# 3. Aspect ratio pie (top middle-right)
ax3 = fig.add_subplot(gs[0, 2])
aspect_counts = df_images['aspect_category'].value_counts()
ax3.pie(aspect_counts.values, labels=aspect_counts.index, autopct='%1.0f%%',
        colors=['#ff6b6b', '#feca57', '#4ecdc4'], startangle=90)
ax3.set_title('Aspect Ratios', fontweight='bold', fontsize=12)

# 4. Quality metrics (top right)
ax4 = fig.add_subplot(gs[0, 3])
quality_scores = ['Good', 'Blurry', 'Low Contrast', 'Corrupt']
quality_vals = [
    len(all_images) - len(blurry_images) - len(df_stats[df_stats['mean_contrast'] < 50]) - len(corrupted),
    len(blurry_images),
    len(df_stats[df_stats['mean_contrast'] < 50]),
    len(corrupted)
]
ax4.barh(quality_scores, quality_vals, color=['green', 'orange', 'yellow', 'red'], alpha=0.7)
ax4.set_title('Quality Distribution', fontweight='bold', fontsize=12)
ax4.set_xlabel('Count')

# 5. Sample images grid (middle row)
ax5 = fig.add_subplot(gs[1, :2])
sample_imgs = df_images['filepath'].tolist()[:6]
for idx, img_path in enumerate(sample_imgs):
    try:
        img = Image.open(img_path)
        # Create inset for each image
        inset = ax5.inset_axes([idx % 3 * 0.33, 0.5 - (idx // 3) * 0.5, 0.3, 0.45])
        inset.imshow(img)
        inset.axis('off')
        inset.set_title(os.path.basename(img_path)[:15], fontsize=8)
    except:
        pass
ax5.set_title('Sample Images', fontweight='bold', fontsize=14, pad=10)
ax5.axis('off')

# 6. RGB distribution (middle right)
ax6 = fig.add_subplot(gs[1, 2:])
x_pos = np.arange(len(df_stats))
width = 0.25
ax6.bar(x_pos - width, df_stats['mean_r'], width, label='Red', color='red', alpha=0.7)
ax6.bar(x_pos, df_stats['mean_g'], width, label='Green', color='green', alpha=0.7)
ax6.bar(x_pos + width, df_stats['mean_b'], width, label='Blue', color='blue', alpha=0.7)
ax6.set_title('RGB Channel Means', fontweight='bold', fontsize=14)
ax6.set_xlabel('Image Index')
ax6.legend()

# 7. Brightness vs Contrast (bottom left)
ax7 = fig.add_subplot(gs[2, 0:2])
scatter = ax7.scatter(df_stats['mean_rgb'], df_stats['mean_contrast'], 
                     c=df_images['megapixels'][:len(df_stats)], cmap='plasma', 
                     s=150, edgecolors='black', alpha=0.7)
ax7.set_xlabel('Mean Brightness')
ax7.set_ylabel('Contrast')
ax7.set_title('Brightness vs Contrast (color=resolution)', fontweight='bold', fontsize=14)
plt.colorbar(scatter, ax=ax7, label='Megapixels')

# 8. File size vs resolution (bottom right)
ax8 = fig.add_subplot(gs[2, 2:])
ax8.scatter(df_images['megapixels'], df_images['file_size_kb'], 
           c=df_images['aspect_ratio'], cmap='coolwarm', s=150, edgecolors='black', alpha=0.7)
ax8.set_xlabel('Resolution (Megapixels)')
ax8.set_ylabel('File Size (KB)')
ax8.set_title('Resolution vs File Size', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

print("\n? Executive dashboard generated!")

### Final Insight: Production Readiness

This comprehensive EDA reveals that our dataset is **production-ready** with minor preprocessing needs:

1. **Diverse but manageable** Good variety in content and resolution without extreme outliers
2. **Quality concerns identified** Specific files flagged for review or removal
3. **Clear preprocessing path** Standardization to RGB, resize to 224x224, normalize using dataset statistics
4. **No critical blockers** No severe class imbalance or corruption issues

Remember: **Garbage in, garbage out**. The time invested in thorough EDA prevents costly retraining and debugging later. Always visualize your data before modeling! 

## Hands-On Exercises  

Now it's your turn to apply these image EDA techniques! Complete these exercises to master image data exploration.

### Exercise 1: Resolution Analysis
Analyze the distribution of image resolutions in detail. Calculate the percentage of images that fall into standard categories: thumbnail (<0.1MP), low (0.1-1MP), medium (1-5MP), high (5-10MP), and ultra-high (>10MP). Create a visualization showing the cumulative distribution of resolutions and identify any outliers using statistical methods.

### Exercise 2: Color Channel Comparison
Select 5 images from the dataset and create a detailed comparison of their RGB channel histograms. Calculate the correlation between channels for each image. Which images have high correlation (similar patterns across channels) vs. low correlation (distinct color patterns)? What does this tell you about the image content?

### Exercise 3: Blur Detection Algorithm
Implement an improved blur detection algorithm that uses both Laplacian variance and FFT (Fast Fourier Transform) based methods. Compare the results between the two approaches. Which method is more reliable for different types of images (natural scenes vs. graphics vs. text)? Create a visualization showing blur scores across the dataset.

### Exercise 4: Duplicate Detection System
Build a robust duplicate detection system using perceptual hashing (phash) instead of simple file properties. Implement a function that can find near-duplicate images even if they have been slightly resized or compressed. Test it on the dataset and report how many duplicates or near-duplicates you find.

### Exercise 5: Aspect Ratio Impact Analysis
Analyze how aspect ratio affects image content. For extreme aspect ratios (very wide >2:1 or very tall <0.5:1), visualize what content is typically lost when forcing these into square crops. Propose a smart cropping strategy that preserves the most important regions of the image.

### Exercise 6: Batch Statistics Calculator
Create a comprehensive batch statistics calculator that processes an entire folder and computes: mean image, standard deviation image, minimum pixel values, and maximum pixel values across the dataset. These "dataset statistics images" are crucial for advanced normalization techniques.

### Exercise 7: Quality Scoring System
Develop a composite quality scoring system that combines multiple metrics: sharpness, contrast, brightness balance, and noise estimation. Assign weights to each metric and calculate an overall quality score (0-100) for each image. Identify the top 10% highest and lowest quality images in the dataset.

### Exercise 8: Color Space Analysis
Convert a subset of images to different color spaces (HSV, LAB, YCbCr) and analyze the distributions in these alternative spaces. Compare the histograms across color spaces. Which color space shows the best separation between different image categories? This informs preprocessing choices for specific computer vision tasks.

### Exercise 9: Reusable EDA Pipeline
Design a complete, reusable ImageEDA class that encapsulates all the analysis techniques from this notebook. The class should have methods for: loading, basic stats, color analysis, quality checks, and report generation. It should be configurable (e.g., enable/disable certain analyses) and export results to both JSON and visual PDF reports.

### Exercise 10: Preprocessing Recommendation Engine
Based on all the EDA findings, build a "preprocessing recommendation engine" that automatically suggests optimal preprocessing steps for a given dataset. The engine should analyze the dataset characteristics and output a tailored preprocessing pipeline configuration (resize strategy, normalization values, augmentation recommendations, quality filters).

## Solutions & Key Insights (Review After Attempting)  

### Exercise 1 Solution: Resolution Categories

**Key Insight**: Resolution distribution heavily influences batch size and memory requirements. High-resolution images (>5MP) require significant GPU memory and may need downsampling or patch-based processing. The cumulative distribution helps identify the optimal target resolution that preserves 90% of image information while maintaining computational efficiency.

In [ ]:
# Exercise 1 Solution
def categorize_resolution_detailed(mp):
    if mp < 0.1:
        return 'Thumbnail (<0.1MP)'
    elif mp < 1:
        return 'Low (0.1-1MP)'
    elif mp < 5:
        return 'Medium (1-5MP)'
    elif mp < 10:
        return 'High (5-10MP)'
    else:
        return 'Ultra (>10MP)'

df_images['res_category'] = df_images['megapixels'].apply(categorize_resolution_detailed)
res_dist = df_images['res_category'].value_counts()

print("RESOLUTION DISTRIBUTION:")
for cat, count in res_dist.items():
    pct = count / len(df_images) * 100
    print(f"   {cat:<20}: {count:3d} ({pct:5.1f}%)")

# Cumulative distribution
sorted_mp = np.sort(df_images['megapixels'])
cumulative = np.arange(1, len(sorted_mp) + 1) / len(sorted_mp) * 100

plt.figure(figsize=(10, 6))
plt.plot(sorted_mp, cumulative, linewidth=2, color='blue')
plt.axhline(90, color='red', linestyle='--', label='90th percentile')
plt.axvline(np.percentile(sorted_mp, 90), color='red', linestyle='--', alpha=0.5)
plt.xlabel('Megapixels')
plt.ylabel('Cumulative Percentage')
plt.title('Cumulative Resolution Distribution')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"\n90th percentile resolution: {np.percentile(sorted_mp, 90):.2f} MP")
print(f"   Suggested target: {int(np.sqrt(np.percentile(sorted_mp, 90) * 1_000_000))}x{int(np.sqrt(np.percentile(sorted_mp, 90) * 1_000_000))}")

### Exercise 2 Solution: Channel Correlation

**Key Insight**: High correlation between RGB channels (correlation > 0.9) indicates grayscale-like or desaturated images, while low correlation suggests vibrant, colorful content. This metric helps identify images that might benefit from color augmentation or conversion to grayscale for efficiency. Medical images often show high correlation, while nature photography shows low correlation.

In [ ]:
# Exercise 2 Solution
def calculate_channel_correlation(image_path):
    """Calculate correlation between RGB channels"""
    with Image.open(image_path) as img:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        arr = np.array(img)
        
        r, g, b = arr[:,:,0].flatten(), arr[:,:,1].flatten(), arr[:,:,2].flatten()
        
        corr_rg = np.corrcoef(r, g)[0,1]
        corr_rb = np.corrcoef(r, b)[0,1]
        corr_gb = np.corrcoef(g, b)[0,1]
        
        return {
            'rg': corr_rg,
            'rb': corr_rb,
            'gb': corr_gb,
            'mean_corr': np.mean([corr_rg, corr_rb, corr_gb])
        }

# Analyze sample images
sample_files = df_images['filepath'].tolist()[:5]
correlations = []

for img_path in sample_files:
    corr = calculate_channel_correlation(img_path)
    corr['filename'] = os.path.basename(img_path)
    correlations.append(corr)
    print(f"{os.path.basename(img_path)}:")
    print(f"   R-G: {corr['rg']:.3f}, R-B: {corr['rb']:.3f}, G-B: {corr['gb']:.3f}")
    print(f"   Mean: {corr['mean_corr']:.3f} ({'High' if corr['mean_corr'] > 0.9 else 'Moderate' if corr['mean_corr'] > 0.7 else 'Low'} correlation)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
corr_matrix = np.array([[corr['mean_corr']] for corr in correlations])
im = axes[0].imshow(corr_matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0].set_xticks([0])
axes[0].set_xticklabels(['Mean Correlation'])
axes[0].set_yticks(range(len(correlations)))
axes[0].set_yticklabels([corr['filename'][:10] for corr in correlations], rotation=0)
axes[0].set_title('Channel Correlation by Image')
plt.colorbar(im, ax=axes[0])

# Bar chart
x_pos = np.arange(len(correlations))
axes[1].bar(x_pos, [c['mean_corr'] for c in correlations], 
           color=['green' if c['mean_corr'] > 0.9 else 'orange' if c['mean_corr'] > 0.7 else 'red' 
                  for c in correlations])
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([c['filename'][:10] for c in correlations], rotation=45)
axes[1].set_ylabel('Mean Correlation')
axes[1].set_title('RGB Channel Correlation')
axes[1].axhline(0.9, color='green', linestyle='--', alpha=0.5, label='High correlation')
axes[1].axhline(0.7, color='orange', linestyle='--', alpha=0.5, label='Moderate')
axes[1].legend()

plt.tight_layout()
plt.show()

### Exercise 3 Solution: Advanced Blur Detection

**Key Insight**: FFT-based blur detection is superior for detecting motion blur and out-of-focus blur in natural images, while Laplacian variance works well for general sharpness. The combination of both methods provides robust quality assessment. FFT reveals frequency domain information sharp images have high-frequency components, while blurry images lack them.

In [ ]:
# Exercise 3 Solution
from scipy import fftpack

def detect_blur_fft(image_path, threshold=10):
    """Detect blur using FFT high-frequency analysis"""
    with Image.open(image_path) as img:
        if img.mode != 'L':
            img = img.convert('L')
        arr = np.array(img)
        
        # FFT
        fft = fftpack.fft2(arr)
        fft_shift = fftpack.fftshift(fft)
        magnitude = np.abs(fft_shift)
        
        # Calculate high-frequency energy ratio
        h, w = magnitude.shape
        center_h, center_w = h // 2, w // 2
        
        # Create mask for high frequencies (outer 30% of spectrum)
        y, x = np.ogrid[:h, :w]
        mask = (x - center_w)**2 + (y - center_h)**2 > (min(h, w) * 0.3)**2
        
        high_freq_energy = np.sum(magnitude[mask])
        total_energy = np.sum(magnitude)
        ratio = high_freq_energy / total_energy * 100
        
        return ratio < threshold, ratio

# Compare methods
print("COMPARING BLUR DETECTION METHODS:")
print("=" * 60)

blur_comparison = []
for img_path in all_images[:8]:  # Test on subset
    # Laplacian method
    is_blur_lap, var_lap = detect_blur(img_path)
    
    # FFT method
    is_blur_fft, ratio_fft = detect_blur_fft(img_path)
    
    blur_comparison.append({
        'filename': os.path.basename(img_path),
        'laplacian_var': var_lap,
        'fft_ratio': ratio_fft,
        'blur_lap': is_blur_lap,
        'blur_fft': is_blur_fft
    })
    
    print(f"{os.path.basename(img_path)[:20]:<20} | Lap: {var_lap:8.1f} ({'Blur' if is_blur_lap else 'Sharp'}) | FFT: {ratio_fft:5.2f}% ({'Blur' if is_blur_fft else 'Sharp'})")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lap_vars = [b['laplacian_var'] for b in blur_comparison]
fft_ratios = [b['fft_ratio'] for b in blur_comparison]
labels = [b['filename'][:8] for b in blur_comparison]

axes[0].scatter(lap_vars, fft_ratios, s=150, alpha=0.7, edgecolors='black')
for i, label in enumerate(labels):
    axes[0].annotate(label, (lap_vars[i], fft_ratios[i]), fontsize=8, ha='center')
axes[0].set_xlabel('Laplacian Variance')
axes[0].set_ylabel('FFT High-Freq Ratio (%)')
axes[0].set_title('Blur Detection: Laplacian vs FFT')
axes[0].axhline(10, color='red', linestyle='--', alpha=0.5, label='FFT Threshold')
axes[0].axvline(100, color='blue', linestyle='--', alpha=0.5, label='Laplacian Threshold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Agreement chart
agreements = [b['blur_lap'] == b['blur_fft'] for b in blur_comparison]
axes[1].pie([sum(agreements), len(agreements)-sum(agreements)], 
           labels=['Agree', 'Disagree'], autopct='%1.0f%%',
           colors=['green', 'red'], startangle=90)
axes[1].set_title('Method Agreement')

plt.tight_layout()
plt.show()

### Exercise 4 Solution: Perceptual Hashing

**Key Insight**: Perceptual hashing (pHash) is robust against minor modifications like compression, resizing, and slight color adjustments. Unlike cryptographic hashes (MD5), pHash generates similar hashes for similar images, enabling detection of near-duplicates. This is crucial for cleaning datasets where the same image might appear in different formats or qualities.

In [ ]:
# Exercise 4 Solution
import hashlib

def phash(image_path, hash_size=8):
    """Calculate perceptual hash (pHash) of an image"""
    with Image.open(image_path) as img:
        # Convert to grayscale and resize
        img = img.convert('L').resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
        pixels = np.array(img)
        
        # Compute difference hash (dHash)
        diff = []
        for row in range(hash_size):
            for col in range(hash_size):
                diff.append(pixels[row, col] > pixels[row, col + 1])
        
        # Convert to hex
        decimal_value = sum(bit << i for i, bit in enumerate(diff))
        return hex(decimal_value)[2:].zfill(hash_size * 2)

def hamming_distance(hash1, hash2):
    """Calculate Hamming distance between two hashes"""
    if len(hash1) != len(hash2):
        return float('inf')
    return sum(c1 != c2 for c1, c2 in zip(hash1, hash2))

def find_similar_images(image_paths, threshold=5):
    """Find similar images using pHash"""
    hashes = {}
    similar_groups = []
    
    for path in image_paths:
        try:
            img_hash = phash(path)
            
            # Check against existing hashes
            found_group = False
            for group in similar_groups:
                if hamming_distance(img_hash, group['hash']) <= threshold:
                    group['images'].append(path)
                    found_group = True
                    break
            
            if not found_group:
                similar_groups.append({'hash': img_hash, 'images': [path]})
                
        except Exception as e:
            print(f"Error processing {path}: {e}")
    
    # Return groups with more than 1 image
    return [g for g in similar_groups if len(g['images']) > 1]

# Test on dataset
print("PERCEPTUAL HASH DUPLICATE DETECTION:")
print("=" * 60)

similar = find_similar_images(all_images, threshold=8)

if similar:
    print(f"Found {len(similar)} groups of similar images:")
    for i, group in enumerate(similar, 1):
        print(f"\nGroup {i} (hash: {group['hash'][:16]}...):")
        for img_path in group['images']:
            print(f"   - {os.path.basename(img_path)}")
else:
    print("No similar images found (this is expected with diverse test images)")
    print("\nTo test, create a duplicate by copying and slightly modifying an image:")
    if all_images:
        # Create a test duplicate
        test_orig = all_images[0]
        test_dup = test_orig.replace('.jpg', '_copy.jpg')
        if os.path.exists(test_orig) and not os.path.exists(test_dup):
            with Image.open(test_orig) as img:
                # Slight modification
                img = img.resize((img.width-10, img.height-10))
                img.save(test_dup, quality=90)
            print(f"Created test duplicate: {test_dup}")
            
            # Test again
            similar = find_similar_images([test_orig, test_dup], threshold=8)
            if similar:
                print(f"? Successfully detected duplicate pair!")
                print(f"   Hamming distance: {hamming_distance(phash(test_orig), phash(test_dup))}")

### Exercise 5 Solution: Smart Cropping Strategy

**Key Insight**: Simple center cropping can remove important content in extreme aspect ratios. A smarter approach uses saliency detection or edge density to identify important regions. For portraits, face detection should guide cropping. For general images, the "rule of thirds" intersection points often contain important content. Implementing smart crop preserves semantic information that naive crops destroy.

In [ ]:
# Exercise 5 Solution
def smart_crop(image_path, target_size=(224, 224)):
    """
    Smart cropping strategy that attempts to preserve important content
    """
    with Image.open(image_path) as img:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        orig_w, orig_h = img.size
        target_w, target_h = target_size
        
        # Calculate crop box
        if orig_w / orig_h > target_w / target_h:
            # Image is wider than target, crop width
            new_w = int(orig_h * target_w / target_h)
            left = (orig_w - new_w) // 2
            crop_box = (left, 0, left + new_w, orig_h)
        else:
            # Image is taller than target, crop height
            new_h = int(orig_w * target_h / target_w)
            top = (orig_h - new_h) // 2
            crop_box = (0, top, orig_w, top + new_h)
        
        cropped = img.crop(crop_box)
        resized = cropped.resize(target_size, Image.Resampling.LANCZOS)
        
        return resized, crop_box

def visualize_cropping_impact(image_path):
    """Visualize different cropping strategies"""
    with Image.open(image_path) as img:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        # Original
        orig_w, orig_h = img.size
        
        # Strategy 1: Center crop
        min_dim = min(orig_w, orig_h)
        left = (orig_w - min_dim) // 2
        top = (orig_h - min_dim) // 2
        center_crop = img.crop((left, top, left + min_dim, top + min_dim))
        
        # Strategy 2: Smart crop (maintain aspect ratio)
        smart_cropped, _ = smart_crop(image_path, (224, 224))
        
        # Strategy 3: Letterbox (pad to square)
        max_dim = max(orig_w, orig_h)
        letterbox = Image.new('RGB', (max_dim, max_dim), (128, 128, 128))
        offset_x = (max_dim - orig_w) // 2
        offset_y = (max_dim - orig_h) // 2
        letterbox.paste(img, (offset_x, offset_y))
        
        return img, center_crop, smart_cropped, letterbox

# Find an extreme aspect ratio image
extreme_images = df_images[(df_images['aspect_ratio'] > 2) | (df_images['aspect_ratio'] < 0.5)]
if len(extreme_images) > 0:
    test_img_path = extreme_images.iloc[0]['filepath']
    
    orig, center, smart, letterbox = visualize_cropping_impact(test_img_path)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(orig)
    axes[0].set_title(f'Original\n{orig.size[0]}x{orig.size[1]}', fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(center)
    axes[1].set_title('Center Crop\n(Square)', fontweight='bold')
    axes[1].axis('off')
    
    axes[2].imshow(smart)
    axes[2].set_title('Smart Crop\n(Aspect Preserved)', fontweight='bold')
    axes[2].axis('off')
    
    axes[3].imshow(letterbox)
    axes[3].set_title('Letterbox\n(Padded)', fontweight='bold')
    axes[3].axis('off')
    
    fig.suptitle('Cropping Strategy Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nComparison for {os.path.basename(test_img_path)}:")
    print(f"   Original aspect ratio: {orig.size[0]/orig.size[1]:.2f}")
    print(f"   Center crop loses: {100 - (center.size[0]*center.size[1])/(orig.size[0]*orig.size[1])*100:.1f}% of pixels")
    print(f"   Recommendation: {'Smart crop' if orig.size[0]/orig.size[1] > 2 or orig.size[0]/orig.size[1] < 0.5 else 'Center crop'}")
else:
    print("No extreme aspect ratio images found in dataset")

### Exercise 6 Solution: Dataset Statistics Images

**Key Insight**: Computing the mean and standard deviation images across the dataset reveals typical patterns and variations. The mean image shows the "average" content (often blurry but reveals dominant colors and central composition), while the std image highlights regions of high variability (usually edges and details). These are essential for proper normalization using dataset-specific statistics rather than ImageNet defaults often improves model convergence, especially for specialized domains like medical imaging or satellite photos.

In [ ]:
# Exercise 6 Solution
def compute_dataset_statistics(image_paths, target_size=(64, 64), max_images=100):
    """
    Compute mean and std images across the dataset
    """
    print(f"Computing statistics for up to {max_images} images...")
    
    # Initialize accumulator
    sum_img = np.zeros((*target_size, 3), dtype=np.float64)
    sum_sq_img = np.zeros((*target_size, 3), dtype=np.float64)
    count = 0
    
    for img_path in image_paths[:max_images]:
        try:
            with Image.open(img_path) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                
                # Resize to target
                img_resized = img.resize(target_size, Image.Resampling.LANCZOS)
                img_array = np.array(img_resized, dtype=np.float64) / 255.0
                
                sum_img += img_array
                sum_sq_img += img_array ** 2
                count += 1
                
                if count % 20 == 0:
                    print(f"   Processed {count} images...")
                    
        except Exception as e:
            print(f"   Error with {img_path}: {e}")
            continue
    
    # Calculate statistics
    mean_img = sum_img / count
    std_img = np.sqrt((sum_sq_img / count) - (mean_img ** 2))
    
    print(f"? Computed statistics from {count} images")
    
    return mean_img, std_img, count

# Compute statistics
mean_image, std_image, n_samples = compute_dataset_statistics(all_images, target_size=(64, 64), max_images=50)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(mean_image)
axes[0].set_title(f'Mean Image\n(n={n_samples})', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(std_image)
axes[1].set_title('Std Dev Image\n(High = Variable)', fontweight='bold')
axes[1].axis('off')

# Channel-wise statistics
channels = ['Red', 'Green', 'Blue']
x_pos = np.arange(len(channels))
mean_values = [mean_image[:,:,i].mean() for i in range(3)]
std_values = [std_image[:,:,i].mean() for i in range(3)]

axes[2].bar(x_pos - 0.2, mean_values, 0.4, label='Mean', color='skyblue', alpha=0.8)
axes[2].bar(x_pos + 0.2, std_values, 0.4, label='Std Dev', color='lightcoral', alpha=0.8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(channels)
axes[2].set_ylabel('Value')
axes[2].set_title('Channel-wise Statistics', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDataset Normalization Values:")
print(f"   Mean: [{mean_image[:,:,0].mean():.4f}, {mean_image[:,:,1].mean():.4f}, {mean_image[:,:,2].mean():.4f}]")
print(f"   Std:  [{std_image[:,:,0].mean():.4f}, {std_image[:,:,1].mean():.4f}, {std_image[:,:,2].mean():.4f}]")
print(f"\nUse these values in transforms.Normalize() for best results!")

### Exercise 7 Solution: Composite Quality Score

**Key Insight**: A composite quality score combining multiple metrics provides a robust way to flag problematic images. Sharpness (FFT), contrast, exposure balance, and color saturation each contribute to perceived quality. Weighted scoring allows customization for specific use cases medical imaging prioritizes sharpness, while artistic content might prioritize color. Automated quality filtering based on these scores can significantly improve training data quality.

In [ ]:
# Exercise 7 Solution
def calculate_quality_score(image_path):
    """
    Calculate composite quality score (0-100)
    """
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            arr = np.array(img)
            
            # 1. Sharpness (30% weight) - using FFT
            gray = np.array(Image.fromarray(arr).convert('L'))
            fft = np.fft.fft2(gray)
            fft_shift = np.fft.fftshift(fft)
            magnitude = np.abs(fft_shift)
            h, w = magnitude.shape
            center_h, center_w = h // 2, w // 2
            y, x = np.ogrid[:h, :w]
            mask = (x - center_w)**2 + (y - center_h)**2 > (min(h, w) * 0.3)**2
            high_freq_ratio = np.sum(magnitude[mask]) / np.sum(magnitude) * 100
            sharpness_score = min(high_freq_ratio * 5, 100)  # Scale to 0-100
            
            # 2. Contrast (25% weight)
            contrast = np.std(gray)
            contrast_score = min(contrast / 255 * 100 * 2, 100)
            
            # 3. Exposure balance (25% weight) - how close to middle gray
            mean_brightness = np.mean(gray)
            exposure_score = 100 - abs(mean_brightness - 128) / 128 * 100
            
            # 4. Color balance (20% weight) - not too saturated or washed out
            r, g, b = arr[:,:,0], arr[:,:,1], arr[:,:,2]
            color_variance = np.std([np.mean(r), np.mean(g), np.mean(b)])
            color_score = 100 - min(color_variance / 255 * 100, 100)
            
            # Composite score
            weights = {'sharpness': 0.30, 'contrast': 0.25, 'exposure': 0.25, 'color': 0.20}
            total_score = (
                sharpness_score * weights['sharpness'] +
                contrast_score * weights['contrast'] +
                exposure_score * weights['exposure'] +
                color_score * weights['color']
            )
            
            return {
                'total': total_score,
                'sharpness': sharpness_score,
                'contrast': contrast_score,
                'exposure': exposure_score,
                'color': color_score
            }
    except Exception as e:
        return {'total': 0, 'error': str(e)}

# Calculate scores for all images
print("CALCULATING QUALITY SCORES...")
quality_scores = []
for img_path in all_images:
    scores = calculate_quality_score(img_path)
    scores['filename'] = os.path.basename(img_path)
    scores['filepath'] = img_path
    quality_scores.append(scores)
    print(f"   {os.path.basename(img_path)[:30]:<30} | Score: {scores['total']:.1f}")

# Sort by quality
quality_scores.sort(key=lambda x: x['total'], reverse=True)

print(f"\nTOP 3 HIGHEST QUALITY:")
for i, q in enumerate(quality_scores[:3], 1):
    print(f"   {i}. {q['filename'][:30]:<30} | Score: {q['total']:.1f}")

print(f"\nTOP 3 LOWEST QUALITY:")
for i, q in enumerate(quality_scores[-3:], 1):
    print(f"   {i}. {q['filename'][:30]:<30} | Score: {q['total']:.1f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Quality Score Analysis', fontsize=16, fontweight='bold')

# Score distribution
all_scores = [q['total'] for q in quality_scores]
axes[0, 0].hist(all_scores, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(70, color='green', linestyle='--', label='Good (70+)')
axes[0, 0].axvline(40, color='red', linestyle='--', label='Poor (<40)')
axes[0, 0].set_xlabel('Quality Score')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Score Distribution')
axes[0, 0].legend()

# Component breakdown
components = ['sharpness', 'contrast', 'exposure', 'color']
comp_means = [np.mean([q[c] for q in quality_scores]) for c in components]
axes[0, 1].bar(components, comp_means, color=['red', 'blue', 'green', 'orange'], alpha=0.7)
axes[0, 1].set_ylabel('Average Score')
axes[0, 1].set_title('Component Averages')
axes[0, 1].set_ylim(0, 100)

# Radar chart for best and worst
angles = np.linspace(0, 2 * np.pi, len(components), endpoint=False).tolist()
angles += angles[:1]

best = quality_scores[0]
worst = quality_scores[-1]

best_values = [best[c] for c in components] + [best[components[0]]]
worst_values = [worst[c] for c in components] + [worst[components[0]]]

ax_radar = fig.add_subplot(2, 2, 3, projection='polar')
ax_radar.plot(angles, best_values, 'o-', linewidth=2, label='Best', color='green')
ax_radar.fill(angles, best_values, alpha=0.25, color='green')
ax_radar.plot(angles, worst_values, 'o-', linewidth=2, label='Worst', color='red')
ax_radar.fill(angles, worst_values, alpha=0.25, color='red')
ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(components)
ax_radar.set_title('Best vs Worst Comparison')
ax_radar.legend()

# Quality vs file size
file_sizes = [os.path.getsize(q['filepath'])/1024 for q in quality_scores]
axes[1, 1].scatter(file_sizes, all_scores, s=100, alpha=0.7, edgecolors='black')
axes[1, 1].set_xlabel('File Size (KB)')
axes[1, 1].set_ylabel('Quality Score')
axes[1, 1].set_title('Quality vs File Size')

plt.tight_layout()
plt.show()

### Exercise 8 Solution: Alternative Color Spaces

**Key Insight**: Different color spaces reveal different patterns. HSV separates color (Hue) from intensity (Value), making it ideal for lighting-invariant analysis. LAB is perceptually uniform and excellent for color difference calculations. YCbCr separates luminance from chrominance, useful for compression and skin detection. For most classification tasks, RGB is sufficient, but for specialized tasks (medical imaging, satellite), alternative color spaces can provide better feature separation.

In [ ]:
# Exercise 8 Solution
from matplotlib.colors import hsv_to_rgb

def analyze_color_spaces(image_path):
    """Analyze image in multiple color spaces"""
    with Image.open(image_path) as img:
        if img.mode != 'RGB':
            img = img.convert('RGB')
        
        rgb_arr = np.array(img) / 255.0
        
        # Convert to HSV
        hsv_arr = plt.matplotlib.colors.rgb_to_hsv(rgb_arr)
        
        # Convert to Grayscale (Luminance)
        gray_arr = np.dot(rgb_arr[...,:3], [0.2989, 0.5870, 0.1140])
        
        return {
            'rgb': rgb_arr,
            'hsv': hsv_arr,
            'gray': gray_arr,
            'filename': os.path.basename(image_path)
        }

# Analyze sample images
sample_img = all_images[0] if all_images else None
if sample_img:
    spaces = analyze_color_spaces(sample_img)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'Color Space Analysis: {spaces["filename"]}', fontsize=16, fontweight='bold')
    
    # RGB
    axes[0, 0].imshow(spaces['rgb'])
    axes[0, 0].set_title('RGB (Original)', fontweight='bold')
    axes[0, 0].axis('off')
    
    # RGB Channels
    for i, (title, color) in enumerate(zip(['Red', 'Green', 'Blue'], ['Reds', 'Greens', 'Blues'])):
        axes[0, i+1].imshow(spaces['rgb'][:,:,i], cmap=color)
        axes[0, i+1].set_title(f'{title} Channel', fontweight='bold')
        axes[0, i+1].axis('off')
    
    # Grayscale
    axes[1, 0].imshow(spaces['gray'], cmap='gray')
    axes[1, 0].set_title('Grayscale (Luminance)', fontweight='bold')
    axes[1, 0].axis('off')
    
    # HSV Channels
    axes[1, 1].imshow(spaces['hsv'][:,:,0], cmap='hsv')  # Hue
    axes[1, 1].set_title('HSV - Hue', fontweight='bold')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(spaces['hsv'][:,:,1], cmap='gray')  # Saturation
    axes[1, 2].set_title('HSV - Saturation', fontweight='bold')
    axes[1, 2].axis('off')
    
    axes[1, 3].imshow(spaces['hsv'][:,:,2], cmap='gray')  # Value
    axes[1, 3].set_title('HSV - Value (Brightness)', fontweight='bold')
    axes[1, 3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Statistical comparison
    print(f"\nCOLOR SPACE STATISTICS:")
    print(f"   RGB Red mean:   {spaces['rgb'][:,:,0].mean():.3f}")
    print(f"   RGB Green mean: {spaces['rgb'][:,:,1].mean():.3f}")
    print(f"   RGB Blue mean:  {spaces['rgb'][:,:,2].mean():.3f}")
    print(f"   HSV Hue mean:   {spaces['hsv'][:,:,0].mean():.3f}")
    print(f"   HSV Sat mean:   {spaces['hsv'][:,:,1].mean():.3f}")
    print(f"   HSV Val mean:   {spaces['hsv'][:,:,2].mean():.3f}")
    print(f"   Gray mean:      {spaces['gray'].mean():.3f}")
    
    print(f"\nINSIGHTS:")
    if spaces['hsv'][:,:,1].mean() < 0.2:
        print(f"   Low saturation Grayscale-like or washed out image")
    if spaces['hsv'][:,:,2].mean() < 0.3:
        print(f"   Low value Dark image, consider brightness augmentation")
    if np.std(spaces['hsv'][:,:,0]) < 0.1:
        print(f"   Low hue variance Monochromatic color scheme")

### Exercise 9 Solution: Reusable ImageEDA Class

**Key Insight**: Encapsulating EDA logic in a reusable class promotes code reuse across projects. Configuration options allow customization without code changes. Export capabilities (JSON for programmatic use, PDF for human review) make the tool versatile for different stakeholders. The class-based design supports inheritance for specialized domains (medical imaging EDA, satellite imagery EDA).

In [ ]:
# Exercise 9 Solution
import json
from datetime import datetime

class ImageEDA:
    """
    Comprehensive Image Exploratory Data Analysis class
    """
    
    def __init__(self, folder_path, config=None):
        self.folder_path = Path(folder_path)
        defaults = {
            'supported_formats': ('.jpg', '.jpeg', '.png', '.bmp', '.tiff'),
            'max_images': None,
            'compute_color_stats': True,
            'compute_quality_metrics': True,
            'target_size': (224, 224)
        }
        self.config = defaults | (config or {})
        self.images = []
        self.results = []
        self.summary = {}
        
    def scan(self):
        """Scan folder for images"""
        print(f"Scanning {self.folder_path}...")
        for ext in self.config['supported_formats']:
            self.images.extend(self.folder_path.glob(f'**/*{ext}'))
            self.images.extend(self.folder_path.glob(f'**/*{ext.upper()}'))
        self.images = list(set(self.images))
        
        if self.config['max_images']:
            self.images = self.images[:self.config['max_images']]
            
        print(f"   Found {len(self.images)} images")
        return self
    
    def analyze(self):
        """Run analysis on all images"""
        print(f"Analyzing {len(self.images)} images...")
        
        for idx, img_path in enumerate(self.images):
            if idx % 10 == 0:
                print(f"   Progress: {idx}/{len(self.images)}")
            
            result = self._analyze_single(img_path)
            self.results.append(result)
        
        self._compute_summary()
        return self
    
    def _analyze_single(self, img_path):
        """Analyze a single image"""
        result = {'filename': img_path.name, 'filepath': str(img_path)}
        
        try:
            with Image.open(img_path) as img:
                # Basic properties
                result.update({
                    'width': img.width,
                    'height': img.height,
                    'mode': img.mode,
                    'format': img.format,
                    'file_size_kb': img_path.stat().st_size / 1024,
                    'aspect_ratio': img.width / img.height,
                    'megapixels': (img.width * img.height) / 1_000_000
                })
                
                # Color statistics
                if self.config['compute_color_stats']:
                    stats_img = img.convert('RGB') if img.mode != 'RGB' else img
                    stat = ImageStat.Stat(stats_img)
                    result.update({
                        'mean_r': stat.mean[0],
                        'mean_g': stat.mean[1],
                        'mean_b': stat.mean[2],
                        'mean_rgb': float(np.mean(stat.mean)),
                        'std_r': stat.stddev[0],
                        'std_g': stat.stddev[1],
                        'std_b': stat.stddev[2],
                        'mean_std': float(np.mean(stat.stddev)),
                        'mean_contrast': float(np.mean([
                            stat.extrema[0][1] - stat.extrema[0][0],
                            stat.extrema[1][1] - stat.extrema[1][0],
                            stat.extrema[2][1] - stat.extrema[2][0]
                        ]))
                    })
                
                # Quality metrics
                if self.config['compute_quality_metrics']:
                    is_blur, blur_var = detect_blur(img_path)
                    result['blur_variance'] = blur_var
                    result['is_blurry'] = is_blur
                    
        except Exception as e:
            result['error'] = str(e)
            
        return result
    
    def _compute_summary(self):
        """Compute dataset summary"""
        df = pd.DataFrame([r for r in self.results if 'error' not in r])
        
        self.summary = {
            'analysis_date': datetime.now().isoformat(),
            'folder': str(self.folder_path),
            'total_images': len(self.results),
            'successful': len(df),
            'failed': len(self.results) - len(df),
            'formats': df['format'].value_counts().to_dict() if 'format' in df.columns else {},
            'modes': df['mode'].value_counts().to_dict() if 'mode' in df.columns else {},
            'avg_resolution': {
                'width': df['width'].mean() if 'width' in df.columns else 0,
                'height': df['height'].mean() if 'height' in df.columns else 0
            },
            'avg_file_size_kb': df['file_size_kb'].mean() if 'file_size_kb' in df.columns else 0,
            'aspect_ratio_range': {
                'min': df['aspect_ratio'].min() if 'aspect_ratio' in df.columns else 0,
                'max': df['aspect_ratio'].max() if 'aspect_ratio' in df.columns else 0
            }
        }
    
    def to_dataframe(self):
        """Return results as DataFrame"""
        return pd.DataFrame(self.results)
    
    def to_json(self, output_path):
        """Export results to JSON"""
        export_data = {
            'summary': self.summary,
            'images': self.results
        }
        with open(output_path, 'w') as f:
            json.dump(export_data, f, indent=2, default=str)
        print(f"Exported to {output_path}")
        
    def generate_report(self):
        """Print summary report"""
        print("\n" + "=" * 60)
        print("IMAGE EDA REPORT")
        print("=" * 60)
        print(f"Folder: {self.summary['folder']}")
        print(f"Date: {self.summary['analysis_date']}")
        print(f"\nTotal Images: {self.summary['total_images']}")
        print(f"Successful: {self.summary['successful']}")
        print(f"Failed: {self.summary['failed']}")
        print(f"\nFormats: {self.summary['formats']}")
        print(f"Avg Resolution: {self.summary['avg_resolution']['width']:.0f} x {self.summary['avg_resolution']['height']:.0f}")
        print(f"Avg File Size: {self.summary['avg_file_size_kb']:.1f} KB")
        print(f"Aspect Ratio: {self.summary['aspect_ratio_range']['min']:.2f} - {self.summary['aspect_ratio_range']['max']:.2f}")

# Demonstrate usage
print("=" * 60)
print("DEMONSTRATING ImageEDA CLASS")
print("=" * 60)

eda = ImageEDA('sample_images', config={'max_images': 20})
eda.scan().analyze().generate_report()
eda.to_json('image_eda_report.json')

print(f"\n? ImageEDA class demonstration complete!")
print(f"   DataFrame shape: {eda.to_dataframe().shape}")
print(f"   Report saved to: image_eda_report.json")

### Exercise 10 Solution: Preprocessing Recommendation Engine

**Key Insight**: An automated recommendation engine eliminates guesswork in preprocessing pipeline design. By analyzing dataset characteristics (resolution distribution, color balance, quality metrics), the engine suggests optimal parameters that would otherwise require trial-and-error. This is especially valuable for MLOps pipelines where new datasets arrive regularly and need rapid assessment. The recommendations balance information preservation with computational efficiency.

In [ ]:
# Exercise 10 Solution
class PreprocessingRecommender:
    """
    Recommends optimal preprocessing steps based on EDA results
    """
    
    def __init__(self, eda_results_df):
        self.df = eda_results_df
        self.recommendations = {}
        
    def analyze(self):
        """Analyze dataset and generate recommendations"""
        if 'error' in self.df.columns:
            valid_df = self.df[self.df['error'].isna()].copy()
        else:
            valid_df = self.df.copy()

        if valid_df.empty:
            raise ValueError('No valid images available for preprocessing analysis.')
        
        # 1. Resize Strategy
        aspect_ratios = valid_df['aspect_ratio']
        ar_std = aspect_ratios.std()
        
        if ar_std < 0.2:
            resize_strategy = "Simple resize to square (dataset has consistent aspect ratios)"
            target_size = int(valid_df['width'].quantile(0.75))
        elif ar_std < 0.5:
            resize_strategy = "RandomResizedCrop with scale (0.8, 1.0) for training, CenterCrop for validation"
            target_size = 224
        else:
            resize_strategy = "Letterbox padding to preserve aspect ratio (extreme variations detected)"
            target_size = 224
            
        self.recommendations['resize'] = {
            'strategy': resize_strategy,
            'target_size': (target_size, target_size),
            'note': f'Aspect ratio std: {ar_std:.2f}'
        }
        
        # 2. Normalization
        if 'mean_r' in valid_df.columns:
            mean = [valid_df['mean_r'].mean()/255, valid_df['mean_g'].mean()/255, valid_df['mean_b'].mean()/255]
            std = [valid_df['std_r'].mean()/255, valid_df['std_g'].mean()/255, valid_df['std_b'].mean()/255]
            
            # If close to ImageNet, suggest ImageNet
            imagenet_mean = [0.485, 0.456, 0.406]
            mean_diff = np.mean([abs(m - i) for m, i in zip(mean, imagenet_mean)])
            
            if mean_diff < 0.1:
                norm_strategy = "Use ImageNet statistics (close match)"
                norm_values = {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}
            else:
                norm_strategy = "Use dataset-specific statistics (significant difference from ImageNet)"
                norm_values = {'mean': mean, 'std': std}
        else:
            norm_strategy = "Use ImageNet statistics (color stats unavailable)"
            norm_values = {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}
            
        self.recommendations['normalization'] = {
            'strategy': norm_strategy,
            'values': norm_values
        }
        
        # 3. Augmentation
        brightness_range = valid_df['mean_rgb'].std() / 255 if 'mean_rgb' in valid_df.columns else 0.1
        
        if brightness_range > 0.15:
            aug_brightness = 0.2
            aug_note = "High brightness variance detected, moderate augmentation recommended"
        else:
            aug_brightness = 0.1
            aug_note = "Consistent brightness, light augmentation sufficient"
            
        self.recommendations['augmentation'] = {
            'RandomHorizontalFlip': True,
            'RandomRotation': 15,
            'ColorJitter': {
                'brightness': aug_brightness,
                'contrast': 0.1,
                'saturation': 0.1
            },
            'note': aug_note
        }
        
        # 4. Quality Filtering
        low_quality = 0
        if 'is_blurry' in valid_df.columns:
            low_quality += valid_df['is_blurry'].sum()
        if 'mean_contrast' in valid_df.columns:
            low_quality += (valid_df['mean_contrast'] < 50).sum()
            
        low_quality_pct = low_quality / len(valid_df) * 100
        
        if low_quality_pct > 10:
            quality_filter = f"Remove {low_quality_pct:.1f}% low quality images (blur variance < 100 or contrast < 50)"
        else:
            quality_filter = "Quality is acceptable, no filtering needed"
            
        self.recommendations['quality_filter'] = quality_filter
        
        # 5. Batch Size
        avg_size_kb = valid_df['file_size_kb'].mean()
        if avg_size_kb < 100:
            batch_size = 64
        elif avg_size_kb < 500:
            batch_size = 32
        else:
            batch_size = 16
            
        self.recommendations['batch_size'] = {
            'recommended': batch_size,
            'note': f'Based on avg file size of {avg_size_kb:.0f} KB'
        }
        
        return self
    
    def generate_config(self):
        """Generate PyTorch/TensorFlow compatible config"""
        config = {
            'input_size': self.recommendations['resize']['target_size'],
            'mean': self.recommendations['normalization']['values']['mean'],
            'std': self.recommendations['normalization']['values']['std'],
            'batch_size': self.recommendations['batch_size']['recommended'],
            'augmentation': self.recommendations['augmentation']
        }
        return config
    
    def print_report(self):
        """Print recommendation report"""
        print("=" * 70)
        print("PREPROCESSING RECOMMENDATION ENGINE")
        print("=" * 70)
        
        for category, details in self.recommendations.items():
            print(f"\n{category.upper()}:")
            if isinstance(details, dict):
                for key, value in details.items():
                    print(f"   {key}: {value}")
            else:
                print(f"   {details}")
        
        print(f"\nPYTORCH TRANSFORMS CONFIG:")
        config = self.generate_config()
        print(f"   transforms.Compose([")
        print(f"       transforms.Resize({config['input_size']}),")
        print(f"       transforms.ToTensor(),")
        print(f"       transforms.Normalize(mean={config['mean']}, std={config['std']})")
        print(f"   ])")

# Generate recommendations
print("=" * 70)
print("GENERATING PREPROCESSING RECOMMENDATIONS")
print("=" * 70)

# Use the EDA results from previous exercise
if 'eda' in dir() and hasattr(eda, 'to_dataframe'):
    recommender = PreprocessingRecommender(eda.to_dataframe())
    recommender.analyze().print_report()
else:
    # Use our main dataset
    combined_df = df_images.merge(df_stats, left_index=True, right_index=True, how='outer')
    recommender = PreprocessingRecommender(combined_df)
    recommender.analyze().print_report()

print("\n? Recommendation engine complete!")